# Movie Rating Prediction — Model Persistence and Prediction Pipeline

**Notebook:** `06_model_persistence.ipynb`
**Modules used:** `src/data_preprocessing.py` (Phase 2), `src/predict.py` (created in this phase)

## 1. Phase 6 Overview

Phase 6 turns the locked model into a **reusable artifact**. No modelling decisions are made here: no tuning, no new
features, no threshold changes, no model selection, and no new evaluation metrics.

**What is saved**

```
raw movie fields (Year, Duration, Genre, Director, Actor 1–3)
        ↓  fitted preprocessing transformer   ── saved
     feature matrix
        ↓  fitted Gradient Boosting model     ── saved
     predicted rating
```

Both parts live in **one** artifact, `models/final_movie_rating_pipeline.joblib`, so a caller never has to reproduce
the preprocessing.

**Locked configuration (from Phase 5, unchanged)**

| Setting | Value |
|---|---|
| Model | `GradientBoostingRegressor` |
| Parameters | `n_estimators=600, learning_rate=0.05, max_depth=4, min_samples_leaf=10, subsample=0.8, random_state=42` |
| Features | `Year`, `Duration`, `Genre`, `Director`, `Actor 1`, `Actor 2`, `Actor 3` |
| Thresholds | director ≥ 10 films, actor ≥ 20 films |
| Excluded | `Votes` (modeling assumption), `Rating` (target), `Name` (identifier only) |

**How this differs from Phase 5**

| Phase | Trained on | Purpose |
|---|---|---|
| 5 | the 6,335 training rows | **final unbiased evaluation** on the untouched 1,584-row test set |
| 6 | **all 7,919 rated rows** | the reusable artifact, so it can use every labelled observation |

The Phase 5 holdout results (**MAE 0.8862, MSE 1.3453, RMSE 1.1599, R² 0.3046**) remain the project's final evaluation
numbers. They describe the Phase 5 model on held-out films. **No metric is computed in this notebook**, because the
artifact here is trained on all rated rows, which leaves no unseen data to evaluate honestly.

## 2. Imports

In [1]:
import hashlib
import json
import platform
import subprocess
import sys
from datetime import date
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "dataset" / "IMDb Movies India.csv"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)
SRC_DIR = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))

import data_preprocessing as dp
import predict as predict_module

RANDOM_STATE = 42

# Locked in Phase 5 — not modified in this notebook
FINAL_PARAMS = dict(n_estimators=600, learning_rate=0.05, max_depth=4, min_samples_leaf=10,
                    subsample=0.8, random_state=RANDOM_STATE)
DIRECTOR_MIN_COUNT, ACTOR_MIN_COUNT = 10, 20
MODEL_PATH = MODEL_DIR / "final_movie_rating_pipeline.joblib"

# Phase 5 holdout results, quoted for reference only; nothing is evaluated in this notebook
PHASE5_HOLDOUT_RESULTS = {"MAE": 0.8862, "MSE": 1.3453, "RMSE": 1.1599, "R²": 0.3046}

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 170)
DIAG = {}
print(f"scikit-learn {sklearn.__version__} | pandas {pd.__version__} | Python {platform.python_version()}")

scikit-learn 1.9.1 | pandas 3.0.6 | Python 3.13.0


## 3. Load and Prepare the Full Labeled Dataset

The artifact is fitted on **every rated row**: the same cleaning as all earlier phases, no split.

In [2]:
PHASE1_SHA256 = "68295a15943a675f38fe8c49af8f5d51b4b833aadb8a4da79da74b0a61c36479"
file_sha256 = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert file_sha256(DATA_PATH) == PHASE1_SHA256

raw = dp.load_raw_data(DATA_PATH)   # encoding="latin-1"
model_df, row_log = dp.build_modeling_frame(raw)
X_full, y_full = dp.get_features_and_target(model_df)

print(pd.Series(row_log).to_string())
print(f"\nLabelled rows for the final artifact: {len(X_full):,}")
print(f"Feature inputs: {list(X_full.columns)}")

DIAG["01 Raw CSV checksum unchanged (matches Phase 1)"] = file_sha256(DATA_PATH) == PHASE1_SHA256
DIAG["02 Rating and Votes absent from the feature inputs"] = not {"Rating", "Votes", "Votes_eda"} & set(X_full.columns)
DIAG["03 All rated rows used for the final artifact"] = len(X_full) == 7919 and y_full.notna().all()

raw_rows                        15509
exact_duplicates_removed            6
exact_duplicates_with_rating        0
rows_after_dedup                15503
unrated_rows_excluded            7584
modeling_rows                    7919

Labelled rows for the final artifact: 7,919
Feature inputs: ['Year', 'Duration_min', 'Genre', 'Director', 'Actor 1', 'Actor 2', 'Actor 3']


## 4. Build the Final Preprocessor

The Phase 2 transformer, with the locked thresholds, fitted on the complete labelled dataset. It learns the duration
statistics, genre vocabulary and director/actor frequencies from exactly that data.

In [3]:
preprocessor = dp.build_feature_transformer(director_min_count=DIRECTOR_MIN_COUNT,
                                            actor_min_count=ACTOR_MIN_COUNT)
preprocessor.fit(X_full)          # no target passed

numeric_step = preprocessor.named_transformers_["numeric"]
director_step = preprocessor.named_transformers_["director"]
actor_step = preprocessor.named_transformers_["actors"]
genre_step = preprocessor.named_transformers_["genre"]
feature_names = list(preprocessor.get_feature_names_out())

print(f"Genre vocabulary            : {len(genre_step.vocabulary_)}")
print(f"Directors counted / one-hot : {len(director_step.name_counts_):,} / {len(director_step.frequent_names_)}")
print(f"Actors counted / multi-hot  : {len(actor_step.name_counts_):,} / {len(actor_step.frequent_names_)}")
print(f"Duration median             : {numeric_step.duration_median_}")
print(f"Total features              : {len(feature_names)}")

Genre vocabulary            : 22
Directors counted / one-hot : 3,139 / 157
Actors counted / multi-hot  : 6,153 / 220
Duration median             : 134.0
Total features              : 413


The feature count is larger than the 322 columns of Phase 5. That is expected: with all 7,919 rated rows instead of
6,335, more directors and actors reach the (unchanged) frequency thresholds. The thresholds themselves are identical.

## 5. Fit the Locked Final Model

Trained once, with exactly the Phase 5 parameters.

In [4]:
final_model = GradientBoostingRegressor(**FINAL_PARAMS)
pipeline = Pipeline([("preprocess", preprocessor), ("model", final_model)])
pipeline.fit(X_full, y_full)

fitted_params = {k: pipeline.named_steps["model"].get_params()[k] for k in FINAL_PARAMS}
print("Fitted model parameters:", fitted_params)
print(f"Training rows: {len(X_full):,}; features seen by the model: {pipeline.named_steps['model'].n_features_in_}")

DIAG["04 Model parameters exactly match the Phase 5 locked configuration"] = fitted_params == FINAL_PARAMS
DIAG["05 Preprocessing thresholds unchanged (director ≥ 10, actor ≥ 20)"] = (
    director_step.min_count == DIRECTOR_MIN_COUNT and actor_step.min_count == ACTOR_MIN_COUNT)
DIAG["06 Model input width equals the transformer output width"] = (
    pipeline.named_steps["model"].n_features_in_ == len(feature_names))

Fitted model parameters: {'n_estimators': 600, 'learning_rate': 0.05, 'max_depth': 4, 'min_samples_leaf': 10, 'subsample': 0.8, 'random_state': 42}
Training rows: 7,919; features seen by the model: 413


## 6. Save the Pipeline

The artifact stores the fitted pipeline together with metadata. The raw dataset is **not** embedded; only its
checksum is recorded.

In [5]:
metadata = {
    "model_name": "Movie Rating Prediction — final pipeline",
    "model_type": "sklearn Pipeline(ColumnTransformer + GradientBoostingRegressor)",
    "project_phase": "Phase 6 artifact — locked Phase 5 configuration retrained on all rated rows",
    "hyperparameters": FINAL_PARAMS,
    "input_fields": list(predict_module.INPUT_FIELDS),
    "feature_columns": list(dp.FEATURE_COLUMNS),
    "n_features_out": len(feature_names),
    "director_min_count": DIRECTOR_MIN_COUNT,
    "actor_min_count": ACTOR_MIN_COUNT,
    "excluded_fields": {"Votes": "modeling assumption: may be unavailable at prediction time",
                        "Rating": "target variable", "Name": "identifier, not a feature"},
    "training_rows": int(len(X_full)),
    "target": dp.TARGET,
    "observed_training_rating_range": [float(y_full.min()), float(y_full.max())],
    "predictions_clipped": False,
    "training_date": date.today().isoformat(),
    "sklearn_version": sklearn.__version__,
    "pandas_version": pd.__version__,
    "python_version": platform.python_version(),
    "source_data_sha256": PHASE1_SHA256,
    "phase5_holdout_results": PHASE5_HOLDOUT_RESULTS,
    "phase5_holdout_note": "Phase 5 final holdout results for the model trained on 6,335 rows; "
                           "this artifact is retrained on all 7,919 rated rows and is not re-evaluated.",
}

joblib.dump({"pipeline": pipeline, "metadata": metadata}, MODEL_PATH, compress=3)
size_mb = MODEL_PATH.stat().st_size / 1024 ** 2
print(f"Saved: {MODEL_PATH.relative_to(PROJECT_ROOT)}")
print(f"Size  : {size_mb:.2f} MB ({MODEL_PATH.stat().st_size:,} bytes)")
DIAG["07 Artifact file exists after saving"] = MODEL_PATH.exists()
DIAG["08 Artifact size is reasonable (< 50 MB, raw data not embedded)"] = size_mb < 50
print(json.dumps({k: v for k, v in metadata.items() if k != "excluded_fields"}, indent=2, default=str))

Saved: models\final_movie_rating_pipeline.joblib
Size  : 0.33 MB (345,420 bytes)
{
  "model_name": "Movie Rating Prediction \u2014 final pipeline",
  "model_type": "sklearn Pipeline(ColumnTransformer + GradientBoostingRegressor)",
  "project_phase": "Phase 6 artifact \u2014 locked Phase 5 configuration retrained on all rated rows",
  "hyperparameters": {
    "n_estimators": 600,
    "learning_rate": 0.05,
    "max_depth": 4,
    "min_samples_leaf": 10,
    "subsample": 0.8,
    "random_state": 42
  },
  "input_fields": [
    "Name",
    "Year",
    "Duration",
    "Genre",
    "Director",
    "Actor 1",
    "Actor 2",
    "Actor 3"
  ],
  "feature_columns": [
    "Year",
    "Duration_min",
    "Genre",
    "Director",
    "Actor 1",
    "Actor 2",
    "Actor 3"
  ],
  "n_features_out": 413,
  "director_min_count": 10,
  "actor_min_count": 20,
  "training_rows": 7919,
  "target": "Rating",
  "observed_training_rating_range": [
    1.1,
    10.0
  ],
  "predictions_clipped": false,
  "t

## 7. Load the Pipeline

`src/predict.py` finds the artifact relative to the project, not the working directory.

In [6]:
predict_module._CACHE.clear()          # ensure a genuine load from disk
artifact = predict_module.load_pipeline()
loaded_pipeline = artifact["pipeline"]

print("Artifact keys:", sorted(artifact))
print("Pipeline steps:", [name for name, _ in loaded_pipeline.steps])
DIAG["09 Artifact contains BOTH the fitted preprocessing and the fitted model"] = (
    [n for n, _ in loaded_pipeline.steps] == ["preprocess", "model"]
    and hasattr(loaded_pipeline.named_steps["preprocess"], "transformers_")
    and hasattr(loaded_pipeline.named_steps["model"], "estimators_"))
DIAG["10 Artifact metadata present and complete"] = all(
    k in artifact["metadata"] for k in ["hyperparameters", "training_rows", "director_min_count", "actor_min_count"])
predict_module.describe_artifact().to_frame()

Artifact keys: ['metadata', 'pipeline']
Pipeline steps: ['preprocess', 'model']


,value
model_name,Movie Rating Prediction — final pipeline
model_type,sklearn Pipeline(ColumnTransformer + GradientB...
project_phase,Phase 6 artifact — locked Phase 5 configuratio...
hyperparameters,"{'n_estimators': 600, 'learning_rate': 0.05, '..."
input_fields,"[Name, Year, Duration, Genre, Director, Actor ..."
feature_columns,"[Year, Duration_min, Genre, Director, Actor 1,..."
n_features_out,413
director_min_count,10
actor_min_count,20
excluded_fields,{'Votes': 'modeling assumption: may be unavail...


## 8. Single-Movie Prediction Test

**Prediction pipeline smoke-test inputs.** The movies below are invented **only to exercise the software**. Their
predicted values are model output for made-up metadata and are **not** real or claimed IMDb ratings.

In [7]:
from predict import predict_movies, predict_rating   # the public interface

smoke_tests = {
    "1. Frequent director and actors": {
        "Name": "Smoke Test: Familiar Names", "Year": "(2015)", "Duration": "150 min", "Genre": "Action, Drama",
        "Director": "Mahesh Bhatt", "Actor 1": "Mithun Chakraborty", "Actor 2": "Dharmendra", "Actor 3": "Shakti Kapoor"},
    "2. Unseen director and actors": {
        "Name": "Smoke Test: Unknown Names", "Year": "(2024)", "Duration": "120 min", "Genre": "Drama, Thriller",
        "Director": "Nonexistent Director XYZ", "Actor 1": "Nonexistent Actor A",
        "Actor 2": "Nonexistent Actor B", "Actor 3": "Nonexistent Actor C"},
    "3. Actor 3 missing": {
        "Name": "Smoke Test: Missing Actor 3", "Year": "(2019)", "Duration": "110 min", "Genre": "Comedy",
        "Director": "David Dhawan", "Actor 1": "Akshay Kumar", "Actor 2": "Paresh Rawal"},
    "4. Multiple genres": {
        "Name": "Smoke Test: Many Genres", "Year": "(2012)", "Duration": "165 min",
        "Genre": "Action, Comedy, Crime", "Director": "Anurag Kashyap", "Actor 1": "Manoj Bajpayee",
        "Actor 2": "Richa Chadha", "Actor 3": "Nawazuddin Siddiqui"},
    "5. Plain strings for Year and Duration": {
        "Name": "Smoke Test: Plain Strings", "Year": "2024", "Duration": "120",
        "Genre": "Documentary", "Director": "Some Director", "Actor 1": "Some Actor"},
}

single = predict_rating(smoke_tests["1. Frequent director and actors"])
print(f"Single prediction: {single:.4f}  (type: {type(single).__name__})")
DIAG["11 Single-movie prediction works and returns a float"] = isinstance(single, float)
DIAG["12 Prediction is finite"] = bool(np.isfinite(single))

Single prediction: 4.8681  (type: float)


In [8]:
results = []
for label, movie in smoke_tests.items():
    value = predict_rating(movie)
    results.append({"smoke-test input": label, "Name": movie["Name"], "Year": movie["Year"],
                    "Duration": movie.get("Duration"), "Genre": movie["Genre"],
                    "Actor 3 supplied": "Actor 3" in movie, "model output": round(value, 3)})
smoke_results = pd.DataFrame(results).set_index("smoke-test input")
smoke_results

,Name,Year,Duration,Genre,Actor 3 supplied,model output
smoke-test input,,,,,,
1. Frequent director and actors,Smoke Test: Familiar Names,(2015),150 min,"Action, Drama",True,4.868
2. Unseen director and actors,Smoke Test: Unknown Names,(2024),120 min,"Drama, Thriller",True,6.783
3. Actor 3 missing,Smoke Test: Missing Actor 3,(2019),110 min,Comedy,False,5.624
4. Multiple genres,Smoke Test: Many Genres,(2012),165 min,"Action, Comedy, Crime",True,7.531
5. Plain strings for Year and Duration,Smoke Test: Plain Strings,2024,120,Documentary,False,8.562


The "model output" column is what the pipeline returns for invented metadata. It is a software check, not a claim
about any real film.

## 9. Unseen Director / Actor Test

A new film has names the model never saw. Phase 2's design handles this: the frequency features become 0 and no
name-specific column is set, rather than failing.

In [9]:
unseen = smoke_tests["2. Unseen director and actors"]
unseen_features = predict_module.prepare_input(unseen)
transformed = loaded_pipeline.named_steps["preprocess"].transform(unseen_features)

print(f"director_count for the unseen director: {transformed['director_count'].iloc[0]}")
print(f"actor_1_count for the unseen actor    : {transformed['actor_1_count'].iloc[0]}")
print(f"Name-specific columns set for this movie: "
      f"{int(transformed[[c for c in transformed.columns if c.startswith(('director=', 'actor='))]].sum(axis=1).iloc[0])}")
value_unseen = predict_rating(unseen)
print(f"Prediction with unseen names: {value_unseen:.4f}")

DIAG["13 Unseen director handled safely (count 0, no crash)"] = (
    transformed["director_count"].iloc[0] == 0 and np.isfinite(value_unseen))
DIAG["14 Unseen actors handled safely (count 0, no crash)"] = bool(
    (transformed[["actor_1_count", "actor_2_count", "actor_3_count"]].iloc[0] == 0).all())
DIAG["15 Transformer is not refitted during prediction"] = (
    director_step.name_counts_ == loaded_pipeline.named_steps["preprocess"].named_transformers_["director"].name_counts_)

director_count for the unseen director: 0
actor_1_count for the unseen actor    : 0
Name-specific columns set for this movie: 0
Prediction with unseen names: 6.7827


## 10. Missing Optional Field Test

Optional fields may be absent. Missing values are passed through to the pipeline's existing missing-value handling;
nothing is invented.

In [10]:
minimal = {"Name": "Smoke Test: Year only", "Year": "(2020)"}
prepared_minimal = predict_module.prepare_input(minimal)
print("Prepared features for a Year-only input:")
display(prepared_minimal)
print(f"Prediction: {predict_rating(minimal):.4f}")

no_actor3 = predict_module.prepare_input(smoke_tests["3. Actor 3 missing"])
transformed_no_a3 = loaded_pipeline.named_steps["preprocess"].transform(no_actor3)
print(f"\nActor 3 missing → actor_3_missing = {transformed_no_a3['actor_3_missing'].iloc[0]}, "
      f"actor_listed = {transformed_no_a3['actor_listed'].iloc[0]}")

DIAG["16 Missing optional actor handled (indicator set, prediction finite)"] = bool(
    transformed_no_a3["actor_3_missing"].iloc[0] == 1 and np.isfinite(predict_rating(smoke_tests["3. Actor 3 missing"])))
DIAG["17 Prediction works with only the required field present"] = bool(np.isfinite(predict_rating(minimal)))

Prepared features for a Year-only input:


,Year,Duration_min,Genre,Director,Actor 1,Actor 2,Actor 3
0,2020.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


Prediction: 6.8856



Actor 3 missing → actor_3_missing = 1, actor_listed = 2


In [11]:
# Rating and Votes are never required, and are ignored when supplied
with_extras = dict(smoke_tests["4. Multiple genres"], Rating=9.9, Votes="123,456")
value_plain = predict_rating(smoke_tests["4. Multiple genres"])
value_extras = predict_rating(with_extras)
print(f"Without Rating/Votes: {value_plain:.6f}")
print(f"With Rating=9.9 and Votes supplied: {value_extras:.6f}")
print("Identical:", value_plain == value_extras)
DIAG["18 Rating is not required and is ignored when supplied"] = value_plain == value_extras
DIAG["19 Votes are not required and are ignored when supplied"] = value_plain == value_extras
DIAG["20 Supplied Rating/Votes never reach the feature matrix"] = not (
    {"Rating", "Votes"} & set(predict_module.prepare_input(with_extras).columns))

try:
    predict_rating({"Name": "No year supplied", "Genre": "Drama"})
    raise AssertionError("missing Year should be rejected")
except ValueError as err:
    print(f"\nMissing required field correctly rejected: {err}")
DIAG["21 Missing required field raises a clear error"] = True

Without Rating/Votes: 7.531394
With Rating=9.9 and Votes supplied: 7.531394
Identical: True

Missing required field correctly rejected: Missing required field(s): ['Year']. Required: ['Year']


## 11. Batch Prediction Test

`predict_movies` accepts a DataFrame of films and returns a copy with a `Predicted Rating` column. `Rating` is not
required.

In [12]:
batch_df = pd.DataFrame([{k: v for k, v in m.items()} for m in smoke_tests.values()])
batch_result = predict_movies(batch_df)

print(f"Input rows: {len(batch_df)} → output rows: {len(batch_result)}")
print(f"Added column: {[c for c in batch_result.columns if c not in batch_df.columns]}")
DIAG["22 Batch prediction returns one prediction per input row"] = len(batch_result) == len(batch_df)
DIAG["23 Batch predictions are all finite numbers"] = bool(np.isfinite(batch_result["Predicted Rating"]).all())
DIAG["24 Batch and single-movie predictions agree"] = bool(np.allclose(
    batch_result["Predicted Rating"].to_numpy(), [predict_rating(m) for m in smoke_tests.values()]))
batch_result[["Name", "Year", "Duration", "Genre", "Director", "Actor 1", "Predicted Rating"]].round(3)

Input rows: 5 → output rows: 5
Added column: ['Predicted Rating']


,Name,Year,Duration,Genre,Director,Actor 1,Predicted Rating
0,Smoke Test: Familiar Names,(2015),150 min,"Action, Drama",Mahesh Bhatt,Mithun Chakraborty,4.868
1,Smoke Test: Unknown Names,(2024),120 min,"Drama, Thriller",Nonexistent Director XYZ,Nonexistent Actor A,6.783
2,Smoke Test: Missing Actor 3,(2019),110 min,Comedy,David Dhawan,Akshay Kumar,5.624
3,Smoke Test: Many Genres,(2012),165 min,"Action, Comedy, Crime",Anurag Kashyap,Manoj Bajpayee,7.531
4,Smoke Test: Plain Strings,2024,120,Documentary,Some Director,Some Actor,8.562


## 12. Artifact Integrity Checks

The reloaded artifact must reproduce the in-memory pipeline exactly, and must carry the locked configuration.

In [13]:
check_features = predict_module.prepare_input(list(smoke_tests.values()))
in_memory = pipeline.predict(check_features)
from_disk = loaded_pipeline.predict(check_features)

print("In-memory pipeline :", np.round(in_memory, 6))
print("Reloaded artifact  :", np.round(from_disk, 6))
DIAG["25 Reloaded artifact reproduces the in-memory pipeline exactly"] = np.array_equal(in_memory, from_disk)
DIAG["26 Saved metadata matches the locked configuration"] = (
    artifact["metadata"]["hyperparameters"] == FINAL_PARAMS
    and artifact["metadata"]["director_min_count"] == DIRECTOR_MIN_COUNT
    and artifact["metadata"]["actor_min_count"] == ACTOR_MIN_COUNT
    and artifact["metadata"]["training_rows"] == len(X_full))
DIAG["27 Feature width of prepared input matches the fitted transformer"] = (
    loaded_pipeline.named_steps["preprocess"].transform(check_features).shape[1] == len(feature_names))
DIAG["28 Artifact does not embed the raw dataset"] = MODEL_PATH.stat().st_size < DATA_PATH.stat().st_size * 20

In-memory pipeline : [4.868133 6.78273  5.624259 7.531394 8.561924]
Reloaded artifact  : [4.868133 6.78273  5.624259 7.531394 8.561924]


## 13. Prediction Reproducibility in a Fresh Python Process

The artifact must work without any notebook state. A **separate** Python process imports `src/predict.py`, loads the
artifact from disk and predicts the same smoke-test inputs. Its results are compared with this session's.

In [14]:
script = f"""
import json, sys
sys.path.insert(0, {str(SRC_DIR)!r})
from predict import predict_rating, load_pipeline
movies = json.loads(sys.argv[1])
print(json.dumps({{"predictions": [predict_rating(m) for m in movies],
                  "training_rows": load_pipeline()["metadata"]["training_rows"]}}))
"""
completed = subprocess.run([sys.executable, "-c", script, json.dumps(list(smoke_tests.values()))],
                           capture_output=True, text=True, cwd=str(PROJECT_ROOT.parent))
print("Fresh process exit code:", completed.returncode)
if completed.returncode != 0:
    print(completed.stderr[-2000:])
fresh = json.loads(completed.stdout)
this_session = [predict_rating(m) for m in smoke_tests.values()]

comparison = pd.DataFrame({"this session": this_session, "fresh process": fresh["predictions"]},
                          index=list(smoke_tests))
comparison["identical"] = comparison["this session"] == comparison["fresh process"]
DIAG["29 Artifact loads in a fresh Python process (no notebook state)"] = completed.returncode == 0
DIAG["30 Fresh-process predictions identical to this session"] = bool(comparison["identical"].all())
comparison.round(6)

Fresh process exit code: 0


,this session,fresh process,identical
1. Frequent director and actors,4.868133,4.868133,True
2. Unseen director and actors,6.782730,6.782730,True
3. Actor 3 missing,5.624259,5.624259,True
4. Multiple genres,7.531394,7.531394,True
5. Plain strings for Year and Duration,8.561924,8.561924,True


## 14. Diagnostics and Phase 6 Summary

In [15]:
DIAG["31 No new evaluation metric computed in this notebook"] = True   # by construction: no scoring call is made
DIAG["32 Raw CSV still unchanged at the end of the notebook"] = file_sha256(DATA_PATH) == PHASE1_SHA256

diag = pd.Series(DIAG, name="passed").sort_index().to_frame()
assert diag["passed"].all(), diag[~diag["passed"]]
print(f"{int(diag['passed'].sum())} / {len(diag)} diagnostics passed")
diag

32 / 32 diagnostics passed


,passed
01 Raw CSV checksum unchanged (matches Phase 1),True
02 Rating and Votes absent from the feature inputs,True
03 All rated rows used for the final artifact,True
04 Model parameters exactly match the Phase 5 locked configuration,True
"05 Preprocessing thresholds unchanged (director ≥ 10, actor ≥ 20)",True
06 Model input width equals the transformer output width,True
07 Artifact file exists after saving,True
"08 Artifact size is reasonable (< 50 MB, raw data not embedded)",True
09 Artifact contains BOTH the fitted preprocessing and the fitted model,True
10 Artifact metadata present and complete,True


In [16]:
print("ARTIFACT")
print(f"  Path            : {MODEL_PATH.relative_to(PROJECT_ROOT)}")
print(f"  Size            : {MODEL_PATH.stat().st_size / 1024 ** 2:.2f} MB")
print(f"  Contents        : fitted preprocessing transformer + fitted model + metadata")
print(f"  Training rows   : {len(X_full):,} (all rated rows)")
print(f"  Features         : {len(feature_names)}")
print(f"  Hyperparameters : {FINAL_PARAMS}")
print(f"  Thresholds      : director ≥ {DIRECTOR_MIN_COUNT}, actor ≥ {ACTOR_MIN_COUNT}; Votes excluded")
print("\nINTERFACE (src/predict.py)")
print("  load_pipeline()            -> {'pipeline': ..., 'metadata': ...}")
print("  predict_rating(movie_dict) -> float")
print("  predict_movies(df)         -> DataFrame with a 'Predicted Rating' column")
print("  describe_artifact()        -> metadata Series")
print(f"\nPhase 5 final holdout results (reference only, NOT recomputed here): {PHASE5_HOLDOUT_RESULTS}")
print(f"Diagnostics passed: {int(diag['passed'].sum())} / {len(diag)}")
print(f"Raw CSV checksum  : {'unchanged' if file_sha256(DATA_PATH) == PHASE1_SHA256 else 'CHANGED'}")

ARTIFACT
  Path            : models\final_movie_rating_pipeline.joblib
  Size            : 0.33 MB
  Contents        : fitted preprocessing transformer + fitted model + metadata
  Training rows   : 7,919 (all rated rows)
  Features         : 413
  Hyperparameters : {'n_estimators': 600, 'learning_rate': 0.05, 'max_depth': 4, 'min_samples_leaf': 10, 'subsample': 0.8, 'random_state': 42}
  Thresholds      : director ≥ 10, actor ≥ 20; Votes excluded

INTERFACE (src/predict.py)
  load_pipeline()            -> {'pipeline': ..., 'metadata': ...}
  predict_rating(movie_dict) -> float
  predict_movies(df)         -> DataFrame with a 'Predicted Rating' column
  describe_artifact()        -> metadata Series

Phase 5 final holdout results (reference only, NOT recomputed here): {'MAE': 0.8862, 'MSE': 1.3453, 'RMSE': 1.1599, 'R²': 0.3046}
Diagnostics passed: 32 / 32
Raw CSV checksum  : unchanged


### Notes and limitations

- **The artifact is trained on all 7,919 rated rows**, so it has no unseen data left and is deliberately **not**
  scored here. The project's final performance numbers stay the **Phase 5 final holdout results**
  (MAE 0.8862, RMSE 1.1599, R² 0.3046), measured on 1,584 films the model had never seen.
- **Predictions are raw model output and are not clipped.** On the smoke-test inputs they stay well inside the
  observed training range of 1.1–10.0, but the model could in principle return a value outside it. Clipping would
  hide that behaviour, so the raw value is returned and this is recorded in the metadata
  (`predictions_clipped: false`).
- **Expected accuracy is what Phase 5 measured:** a typical error around 0.9 rating points, with larger errors for
  films that turn out to be very good or very bad. Predictions are statistical estimates from limited metadata, not
  reliable forecasts for an individual film, and the pipeline is **not production-ready**.
- **Unseen names are safe but uninformative:** a new director or actor gets a frequency of 0 and no name-specific
  column, so the prediction leans on year, duration and genre. Phase 5 found higher errors for directors with little
  or no history.
- **Scope:** the training data is Indian films up to 2021–22. Inputs far outside that scope are extrapolation.
- The artifact stores the fitted objects only. The raw CSV is **not** embedded; only its checksum is recorded.